# 🎬 企業 SOP 商用劇情短片工作站（Colab G4 / A100 / L4｜Wan 2.2 Apache-2.0｜NT$50 硬體預算煞車）

專為**企業商用合規**設計：**100% Apache-2.0 開源授權、無美歐機房地區限制**（不含 MiniMax H3 / Qwen-Image 2.1 / Breeze TTS 2 等非商用或限區模型）。

### 核心特色
1. **首幀鎖角色（I2V）**：上傳你鎖定好 PPE 與臉部的角色首幀圖片，由阿里通義 **Wan 2.2 I2V 14B（Apache-2.0）** 產出高穩定度 720p 實拍感短片，解決 Flow 換臉、PPE 亂變、動作不聽話的問題。
2. **NT$50 硬體預算自動煞車（Hard Budget Guard）**：
   - 從第 1 格執行瞬間開始計時，依自動偵測到的 GPU 型號即時換算已花費新台幣。
   - 一旦達到 `MAX_BUDGET_NTD`（預設 **NT$50**）或預估跑下一支會超標 → **立刻停拍 → 優先打包下載已完成的所有影片 → 15 秒後自動釋放 Colab GPU 停止計費**。
3. **Google Drive 模型快取**：強烈建議開啟 `USE_GOOGLE_DRIVE_CACHE`，第一次下載模型後存雲端，下次開機免花錢等下載。

---
### 三步驟使用方式
1. 切換執行階段為 **`G4`**（最推薦：96GB VRAM 速度最快、單支最省）或 **`A100` / `L4`**。
2. 在第 1 格設定 `MAX_BUDGET_NTD`（預設 `50` 元）與解析度，按 **`執行階段 → 全部執行`**。
3. 第 3 格上傳首幀圖片（`.png/.jpg`）與選填的 `video_jobs.json`；完成或達預算上限時會自動下載 `sop_wan22_clips.zip` 並自動關機停費。

In [ ]:
#@title ⚙️ 1. 參數與 NT$50 硬體預算煞車設定（從本格開始精準計費）
MODEL_VARIANT = "Wan2.2-I2V-A14B (推薦 G4/A100 滿血畫質)" #@param ["Wan2.2-I2V-A14B (推薦 G4/A100 滿血畫質)", "Wan2.2-TI2V-5B (輕量 L4 省點版)"]
RESOLUTION = "720p (1280x720)" #@param ["720p (1280x720)", "480p (832x480)"]
DURATION_SEC = 5 #@param {type:"slider", min:3, max:5, step:1}
INFERENCE_STEPS = 28 #@param {type:"slider", min:15, max:40, step:1}
GUIDANCE_SCALE = 3.5 #@param {type:"number"}
MAX_BUDGET_NTD = 50.0 #@param {type:"number"}
NTD_PER_CU = 3.4 #@param {type:"number"}
AUTO_SHUTDOWN_AFTER_DONE = True #@param {type:"boolean"}
USE_GOOGLE_DRIVE_CACHE = True #@param {type:"boolean"}

import subprocess
import time
from pathlib import Path

SESSION_START_TS = time.time()
WORK_DIR = Path("/content/wan22_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BUDGET_STOPPED = False

# 各 GPU 實測 Colab 運算單元費率 (CU/hr)
GPU_CU_RATES = [
    ("RTX PRO 6000", 8.90),
    ("Blackwell", 8.90),
    ("H100", 11.90),
    ("A100", 5.30),
    ("L4", 1.54),
    ("T4", 1.07),
]

res = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if res.returncode != 0 or not res.stdout.strip():
    raise RuntimeError("⚠️ 未偵測到 GPU！請點「執行階段 → 變更執行階段類型」選擇 G4 / A100 / L4。")

GPU_LINE = res.stdout.strip().splitlines()[0]
GPU_NAME = GPU_LINE.split(",")[0].strip()
VRAM_MB = int("".join(ch for ch in GPU_LINE.split(",")[1] if ch.isdigit()) or "0")
VRAM_GB = round(VRAM_MB / 1024, 1)

CU_PER_HR = 5.30
for kw, rate in GPU_CU_RATES:
    if kw.lower() in GPU_NAME.lower():
        CU_PER_HR = rate
        break

NTD_PER_HR = CU_PER_HR * NTD_PER_CU
MAX_RUNTIME_MIN = (MAX_BUDGET_NTD / NTD_PER_HR) * 60.0

def budget_status(est_next_sec: float = 0.0):
    elapsed_sec = time.time() - SESSION_START_TS
    spent_cu = (elapsed_sec / 3600.0) * CU_PER_HR
    spent_ntd = spent_cu * NTD_PER_CU
    proj_ntd = ((elapsed_sec + est_next_sec) / 3600.0) * NTD_PER_HR
    rem_ntd = max(0.0, MAX_BUDGET_NTD - spent_ntd)
    ok = (spent_ntd < MAX_BUDGET_NTD) and (proj_ntd <= MAX_BUDGET_NTD * 1.02)
    return ok, elapsed_sec, spent_cu, spent_ntd, rem_ntd

print(f"✅ GPU：{GPU_NAME} ({VRAM_GB} GB VRAM)")
print(f"💰 費率：{CU_PER_HR:.2f} CU/hr ≈ NT${NTD_PER_HR:.1f}/小時（每單元 NT${NTD_PER_CU}）")
print(f"🛡️ 預算上限：NT${MAX_BUDGET_NTD:.1f}（最長可跑約 {MAX_RUNTIME_MIN:.1f} 分鐘，達標自動打包下載並關機）")
if "A14B" in MODEL_VARIANT and VRAM_GB < 20:
    print("⚠️ 目前顯存 < 20GB，建議改選「Wan2.2-TI2V-5B」或切換至 G4 / A100 執行階段。")


In [ ]:
#@title 📦 2. 安裝 Diffusers 環境與載入快取（約 1–2 分鐘）
import os
import shutil
import subprocess
import sys
from pathlib import Path

ok, el, cu, ntd, rem = budget_status()
if not ok:
    BUDGET_STOPPED = True
    print(f"🛑 已達預算上限 NT${ntd:.2f}，跳過安裝直接進入結算。")
else:
    MODEL_CACHE = Path("/content/hf_cache")
    if USE_GOOGLE_DRIVE_CACHE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            MODEL_CACHE = Path("/content/drive/MyDrive/AI_Wan22_Cache")
            print("💾 已啟用 Google Drive 模型快取（下次免重下載省單元）：", MODEL_CACHE)
        except Exception as exc:
            print(f"⚠️ 掛載 Google Drive 失敗，改用本機暫存 ({exc})")
    MODEL_CACHE.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(MODEL_CACHE)
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

    pkgs = [
        "diffusers>=0.34.0",
        "transformers>=4.49.0",
        "accelerate>=1.2.0",
        "ftfy",
        "sentencepiece",
        "imageio[ffmpeg]",
        "hf_transfer",
    ]
    print("⏳ 安裝 Wan 2.2 Diffusers 套件...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)
    if shutil.which("ffmpeg") is None:
        subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)

    ok, el, cu, ntd, rem = budget_status()
    print(f"✅ 環境就緒！累計耗時 {el/60:.1f} 分 | 已花費約 NT${ntd:.2f} / 剩餘 NT${rem:.2f}")


In [ ]:
#@title 🖼️ 3. 上傳首幀圖片與（選填）分鏡 video_jobs.json
import json
import shutil
from pathlib import Path

UPLOAD_DIR = WORK_DIR / "uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
JOBS_PATH = WORK_DIR / "video_jobs.json"

DEFAULT_NEG_PROMPT = (
    "blurry, distorted face, extra fingers, missing helmet, changing clothes, "
    "text, watermark, subtitle, logo, low quality, jitter, flicker, cartoon, anime"
)

if not BUDGET_STOPPED:
    print('📤 請上傳：首幀圖片 (.png/.jpg，可多選) ＋ 選填 video_jobs.json（格式：[{"name":"s01","image":"s01.png","prompt":"..."}]）：')
    uploaded_files = []
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname, raw in uploaded.items():
            dst = UPLOAD_DIR / Path(fname).name
            dst.write_bytes(raw)
            uploaded_files.append(dst)
            print("  ✔ 已收到：", dst.name)
    except Exception as exc:
        print("ℹ️ 略過互動上傳：", exc)

    jsons = [p for p in uploaded_files if p.suffix.lower() == ".json"] or list(UPLOAD_DIR.glob("*.json"))
    img_exts = {".png", ".jpg", ".jpeg", ".webp"}
    imgs = sorted([p for p in UPLOAD_DIR.iterdir() if p.suffix.lower() in img_exts])

    if jsons:
        shutil.copy2(jsons[0], JOBS_PATH)
        print("📄 已套用分鏡設定檔：", jsons[0].name)
    elif imgs:
        auto_jobs = []
        for idx, img in enumerate(imgs, start=1):
            auto_jobs.append({
                "name": img.stem,
                "image": img.name,
                "prompt": (
                    "Realistic industrial safety training film footage, smooth slow cinematic camera push-in, "
                    "technician wearing full PPE helmet and safety goggles performs careful inspection at chemical tank farm, "
                    "natural daylight, sharp focus, 24fps film look."
                ),
                "negative_prompt": DEFAULT_NEG_PROMPT,
                "seed": 100 + idx,
            })
        JOBS_PATH.write_text(json.dumps(auto_jobs, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"📄 未上傳 JSON，已自動為 {len(auto_jobs)} 張首幀圖建立工業 SOP 運鏡任務。")
    else:
        raise RuntimeError("⚠️ 至少需要上傳 1 張首幀圖片 (.png/.jpg) 才能啟動 Wan 2.2 I2V！")

    jobs_preview = json.loads(JOBS_PATH.read_text(encoding="utf-8-sig"))
    print(f"🎬 待生成鏡頭數：{len(jobs_preview)} 鏡")


In [ ]:
#@title 🎬 4. GPU 批次生片（每鏡前即時檢查 NT$50 預算煞車）
import gc
import json
import subprocess
import time
from pathlib import Path
import numpy as np
import torch
from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

OUT_DIR = WORK_DIR / "clips"
OUT_DIR.mkdir(parents=True, exist_ok=True)
COST_LOG = []

if not BUDGET_STOPPED:
    model_id = (
        "Wan-AI/Wan2.2-I2V-A14B-Diffusers"
        if "A14B" in MODEL_VARIANT
        else "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
    )
    print(f"⏳ 載入 Apache-2.0 模型：{model_id}（首次下載約 2–4 分鐘，後續讀快取秒載）...")
    t_load = time.time()
    pipe = WanImageToVideoPipeline.from_pretrained(model_id, torch_dtype=torch.bfloat16)
    if VRAM_GB >= 75:
        pipe.to("cuda")
        print(f"🚀 G4 大顯存模式（{VRAM_GB}GB）：全模型常駐顯存加速！載入耗時 {time.time() - t_load:.1f}s")
    else:
        pipe.enable_model_cpu_offload()
        print(f"⚙️ CPU Offload 模式（{VRAM_GB}GB VRAM）：載入耗時 {time.time() - t_load:.1f}s")

    jobs = json.loads((WORK_DIR / "video_jobs.json").read_text(encoding="utf-8-sig"))
    max_area = (720 * 1280) if "720p" in RESOLUTION else (480 * 832)
    num_frames = 81 if DURATION_SEC >= 5 else (65 if DURATION_SEC == 4 else 49)
    avg_clip_sec = 60.0 if VRAM_GB >= 75 else 180.0

    for idx, j in enumerate(jobs, start=1):
        ok, el, cu, ntd, rem = budget_status(est_next_sec=avg_clip_sec * 0.85)
        if not ok:
            BUDGET_STOPPED = True
            print(
                f"\n🛑 觸發 NT${MAX_BUDGET_NTD:.1f} 硬體預算煞車！"
                f"目前累計已花費約 NT${ntd:.2f}（剩餘 NT${rem:.2f}，不足安全跑完第 {idx} 鏡）。"
            )
            print("📦 立刻停止後續生成，直接跳至第 5 格打包下載已完成影片並自動關機！")
            break

        img_path = WORK_DIR / "uploads" / j["image"]
        if not img_path.exists():
            print(f"⚠️ 找不到首幀圖 {img_path.name}，略過 {j['name']}")
            continue

        out_raw = OUT_DIR / f"{j['name']}_16fps.mp4"
        out_final = OUT_DIR / f"{j['name']}.mp4"
        if out_final.exists():
            print(f"⏭️ 已存在 {out_final.name}，跳過")
            continue

        image = load_image(str(img_path))
        aspect_ratio = image.height / image.width
        mod_value = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
        height = max(mod_value, round(np.sqrt(max_area * aspect_ratio)) // mod_value * mod_value)
        width = max(mod_value, round(np.sqrt(max_area / aspect_ratio)) // mod_value * mod_value)
        image = image.resize((width, height))

        seed = int(j.get("seed", 100 + idx))
        gen = torch.Generator(device="cuda").manual_seed(seed)
        print(f"\n🎥 [{idx}/{len(jobs)}] 生成 {j['name']} ({width}x{height}, {num_frames}f, seed={seed}) | 剩餘預算 NT${rem:.2f}")
        t0 = time.time()
        frames = pipe(
            image=image,
            prompt=j["prompt"],
            negative_prompt=j.get("negative_prompt", ""),
            height=height,
            width=width,
            num_frames=num_frames,
            guidance_scale=float(j.get("guidance_scale", GUIDANCE_SCALE)),
            num_inference_steps=int(j.get("steps", INFERENCE_STEPS)),
            generator=gen,
        ).frames[0]
        export_to_video(frames, str(out_raw), fps=16)

        # 轉為標準 24fps H.264 MP4（直接相容本機 FFmpeg SOP 剪接管線）
        subprocess.run([
            "ffmpeg", "-y", "-i", str(out_raw),
            "-vf", "fps=24", "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p",
            str(out_final),
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        out_raw.unlink(missing_ok=True)

        clip_sec = time.time() - t0
        avg_clip_sec = clip_sec if idx == 1 else (avg_clip_sec * 0.5 + clip_sec * 0.5)
        clip_ntd = (clip_sec / 3600.0) * NTD_PER_HR
        _, el_now, cu_now, ntd_now, rem_now = budget_status()
        COST_LOG.append({
            "name": j["name"],
            "file": out_final.name,
            "size": f"{width}x{height}",
            "seed": seed,
            "sec": round(clip_sec, 1),
            "est_clip_ntd": round(clip_ntd, 2),
            "cum_ntd": round(ntd_now, 2),
        })
        print(f"  ✅ 完成 {out_final.name} | 耗時 {clip_sec:.1f}s (單鏡約 NT${clip_ntd:.2f}) | 累計 NT${ntd_now:.2f} / 上限 NT${MAX_BUDGET_NTD:.1f}")

    del pipe
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
#@title 📦 5. 打包下載影片＋費用明細＋自動關機停費（絕不白燒錢）
import json
import time
import zipfile
from pathlib import Path

_, el_tot, cu_tot, ntd_tot, rem_tot = budget_status()
clips = sorted((WORK_DIR / "clips").glob("*.mp4")) if (WORK_DIR / "clips").exists() else []

summary = {
    "gpu": GPU_NAME,
    "vram_gb": VRAM_GB,
    "cu_per_hr": CU_PER_HR,
    "budget_cap_ntd": MAX_BUDGET_NTD,
    "budget_brake_triggered": BUDGET_STOPPED,
    "elapsed_min": round(el_tot / 60.0, 2),
    "est_total_cu": round(cu_tot, 3),
    "est_total_ntd": round(ntd_tot, 2),
    "clips_completed": len(clips),
    "avg_ntd_per_clip": round(ntd_tot / len(clips), 2) if clips else 0.0,
    "details": COST_LOG,
}

REPORT_PATH = WORK_DIR / "cost_report.json"
REPORT_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

ZIP_PATH = Path("/content/sop_wan22_clips.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(REPORT_PATH, "cost_report.json")
    for mp4 in clips:
        zf.write(mp4, f"clips/{mp4.name}")

print("=" * 64)
print(f"🎬 完成鏡頭：{len(clips)} 支 | 總耗時：{summary['elapsed_min']} 分鐘")
print(f"💰 預估總費用：NT${summary['est_total_ntd']:.2f}（{summary['est_total_cu']:.3f} CU） / 上限 NT${MAX_BUDGET_NTD:.1f}")
if clips:
    print(f"📊 平均單鏡成本：NT${summary['avg_ntd_per_clip']:.2f} / 支")
print(f"📦 已打包：{ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024**2:.1f} MB)")
print("=" * 64)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
    print("⬇ 已觸發瀏覽器自動下載 sop_wan22_clips.zip！")
except Exception as exc:
    print(f"💡 成品位置：{ZIP_PATH} ({exc})")

if AUTO_SHUTDOWN_AFTER_DONE or BUDGET_STOPPED:
    print("⏳ 等待 15 秒確保瀏覽器開始下載，接著自動釋放 Colab GPU 停止計費...")
    time.sleep(15)
    try:
        from google.colab import runtime
        print("🔌 正在自動斷開並釋放 GPU 執行階段（停止計費）...")
        runtime.unassign()
    except Exception as exc:
        print(f"⚠️ 請手動點右上角「中斷連線並刪除執行階段」停止計費 ({exc})")
